# Kiểm tra bộ dữ liệu DiverseVul

Kiểm tra các bản ghi C/C++ ở mức hàm để đánh giá mức phù hợp cho bài toán phân loại CWE. Notebook không tự chọn số CWE phổ biến nhất, chính sách nhãn, cách chia dữ liệu cuối cùng, biểu diễn đồ thị hoặc kiến trúc GNN.

## 1. Thiết lập

Xác định mọi đường dẫn từ thư mục gốc repository và cố định seed dùng để lấy mẫu.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.loader import load_dataset
from src.data.diversevul_audit import run_audit, read_optional_metadata, metadata_audit, schema_table
from src.utils.config import load_config

SEED = 105
config = load_config(ROOT / "configs/data/diversevul.yaml")["dataset"]
raw_dir = ROOT / config["local_raw_dir"]
print("Repository:", ROOT)
print("Thư mục dữ liệu thô:", raw_dir)

Repository: D:\PROJECT\software-bug-detection-using-graphs
Thư mục dữ liệu thô: D:\PROJECT\software-bug-detection-using-graphs\data\raw\diversevul


## 2. Nạp dữ liệu

Nạp đầy đủ bản dữ liệu local thực tế. Nếu thiếu dữ liệu, trạng thái bị chặn phải được hiển thị tường minh.

In [2]:
dataset_available = False
blocked_reason = None
try:
    loaded = load_dataset(config["name"], raw_dir)
    raw = loaded.frame
    dataset_available = True
    print("File:", loaded.source_file)
    print("Shape:", raw.shape)
except FileNotFoundError as error:
    blocked_reason = str(error)
    print("BLOCKED:", blocked_reason)

BLOCKED: Thư mục dữ liệu thô DiverseVul không tồn tại: D:\PROJECT\software-bug-detection-using-graphs\data\raw\diversevul. Xem hướng dẫn tại data/raw/README.md.


## 3. Cấu trúc dữ liệu

In các cột, kiểu dữ liệu, kích thước, ánh xạ field, dữ liệu thiếu và ví dụ giới hạn từ dữ liệu thật. Không tự tạo field.

In [3]:
if dataset_available:
    result = run_audit(raw, config)
    print(raw.dtypes)
    print("Các field đã ánh xạ:", result.fields)
    display(result.tables["missing_values"])
else:
    print("Chưa quan sát được schema vì không có file dataset nào được load.")

Chưa quan sát được schema vì không có file dataset nào được load.


## 4. Xác minh thống kê trong paper

Tính lại sáu metric chính và giữ nguyên mọi chênh lệch so với baseline trong paper.

In [4]:
display(result.tables['paper_comparison']) if dataset_available else print('Chưa tính được.')

Chưa tính được.


## 5. Dữ liệu bị thiếu

Kiểm tra missingness của từng cột thô, bao gồm các semantic field không tồn tại.

In [5]:
display(result.tables['missing_values']) if dataset_available else print('Chưa tính được.')

Chưa tính được.


## 6. Mức bao phủ CWE của dữ liệu có lỗ hổng

Phân tích cho bài toán phân loại CWE chỉ sử dụng các bản ghi có lỗ hổng.

In [6]:
display(result.tables['dataset_summary']) if dataset_available else print('Chưa tính được.')

Chưa tính được.


## 7. Phân bố CWE

Tính mức bao phủ theo hàm, dự án, commit và dữ liệu có lỗ hổng đã gắn nhãn cho từng CWE. Bản ghi có nhiều CWE đóng góp vào từng CWE được gắn.

In [7]:
display(result.tables['cwe_distribution'].head(20)) if dataset_available else print('Chưa tính được.')

Chưa tính được.


## 8. Phân tích ngưỡng

Đánh giá các ngưỡng 20/50/100/200 mẫu trên mỗi CWE mà không tự chọn ngưỡng.

In [8]:
display(result.tables['threshold_analysis']) if dataset_available else print('Chưa tính được.')

Chưa tính được.


## 9. Phân tích mẫu có nhiều CWE

Đo số mẫu có 0, 1 hoặc từ 2 CWE trở lên mà không tự chuyển bài toán thành phân loại một nhãn.

In [9]:
if dataset_available:
    display(result.tables["multi_cwe_summary"])
    display(result.tables["cwe_count_distribution"])
    display(result.tables["multi_cwe_examples"])
else:
    print("Chưa tính được.")

Chưa tính được.


## 10. Phân bố dự án

So sánh tổng số hàm, số hàm có lỗ hổng và độ rộng CWE trên từng dự án.

In [10]:
display(result.tables['project_distribution'].head(20)) if dataset_available else print('Chưa tính được.')

Chưa tính được.


## 11. Phân tích CWE theo dự án

Đo số dự án hỗ trợ và mức tập trung vào dự án lớn nhất cho từng CWE.

In [11]:
if dataset_available:
    display(result.tables["cwe_project_distribution"].head(30))
    display(result.tables["project_split_feasibility"])
else:
    print("Chưa tính được.")

Chưa tính được.


## 12. Phân tích dữ liệu trùng lặp

So sánh mã băm của mã nguồn nguyên bản với mã băm sau khi chuẩn hóa bảo thủ ký tự xuống dòng và khoảng trắng cuối dòng, đồng thời đếm xung đột nhãn.

In [12]:
display(result.tables['duplicate_summary']) if dataset_available else print('Chưa tính được.')

Chưa tính được.


## 13. Chất lượng mã nguồn

Báo cáo mã nguồn bị thiếu/rỗng và các phân vị độ dài theo ký tự/dòng. Chủ động không tính token khi chưa chọn bộ tách token.

In [13]:
if dataset_available:
    display(result.tables["source_quality_summary"])
    display(result.tables["function_length_summary"])
    display(result.tables["source_inspection_sample"])
else:
    print("Chưa tính được.")

Chưa tính được.


## 14. Mẫu kiểm tra mức sẵn sàng để tạo đồ thị

Chỉ tạo danh mục nhỏ với seed cố định để thử Joern ở giai đoạn sau; không tạo CPG trong notebook này.

In [14]:
if dataset_available:
    display(result.tables["graph_sample_manifest"])
else:
    print("Không thể tạo manifest khi chưa có dataset thô.")

Không thể tạo manifest khi chưa có dataset thô.


## 15. Kết quả

Sinh tất cả bảng, biểu đồ, danh mục mẫu, kết quả kiểm tra siêu dữ liệu và báo cáo Markdown từ cùng một kết quả đã tính.

In [15]:
import subprocess
completed = subprocess.run(
    [sys.executable, str(ROOT / "scripts/run_diversevul_eda.py")],
    cwd=ROOT,
    check=True,
    text=True,
    capture_output=True,
)
print(completed.stdout.strip())
if dataset_available:
    metadata, _ = read_optional_metadata(raw_dir)
    result.tables["metadata_audit"] = metadata_audit(result.records, metadata)
    print(result.findings)
else:
    print("Không có kết quả EDA nào được tuyên bố; các artifact được sinh đều ghi rõ trạng thái BLOCKED.")

BLOCKED: Thư mục dữ liệu thô DiverseVul không tồn tại: D:\PROJECT\software-bug-detection-using-graphs\data\raw\diversevul. Xem hướng dẫn tại data/raw/README.md.
Không có kết quả EDA nào được tuyên bố; các artifact được sinh đều ghi rõ trạng thái BLOCKED.


## 16. Các quyết định nghiên cứu còn mở

Bằng chứng hỗ trợ nhưng không tự chốt các quyết định dưới đây.

In [16]:
decisions = [
    "Top-K hoặc số sample tối thiểu trên mỗi CWE",
    "cách xử lý single-label, multi-label, hierarchical hoặc ambiguous sample",
    "project-aware split cuối cùng",
    "chính sách xử lý duplicate và label conflict",
    "graph representation và cấu hình Joern",
    "kiến trúc GNN và evaluation protocol",
]
for decision in decisions:
    print("-", decision)

- Top-K hoặc số sample tối thiểu trên mỗi CWE
- cách xử lý single-label, multi-label, hierarchical hoặc ambiguous sample
- project-aware split cuối cùng
- chính sách xử lý duplicate và label conflict
- graph representation và cấu hình Joern
- kiến trúc GNN và evaluation protocol
